In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("earthquake-exploration")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.5.0,com.amazonaws:aws-java-sdk-bundle:1.12.367,org.postgresql:postgresql:42.7.3")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.session.timeZone", "UTC")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 07:29:21 WARN Utils: Your hostname, tammem, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/16 07:29:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/msi/spark/spark-4.2.0-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/msi/.ivy2.5.2/cache
The jars for the packages stored in: /home/msi/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f7110b6b-c0b5-43ce-a3d3-b62e6eb9a71b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelera

In [3]:
from pyspark.sql.types import (
    StructType, StructField, TimestampType, DoubleType,
    StringType, IntegerType
)

earthquake_schema = StructType([
    StructField("time", TimestampType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("depth", DoubleType(), True),
    StructField("mag", DoubleType(), True),
    StructField("magType", StringType(), True),
    StructField("nst", IntegerType(), True),
    StructField("gap", DoubleType(), True),
    StructField("dmin", DoubleType(), True),
    StructField("rms", DoubleType(), True),
    StructField("net", StringType(), True),
    StructField("id", StringType(), True),
    StructField("updated", TimestampType(), True),
    StructField("place", StringType(), True),
    StructField("type", StringType(), True),
    StructField("horizontalError", DoubleType(), True),
    StructField("depthError", DoubleType(), True),
    StructField("magError", DoubleType(), True),
    StructField("magNst", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("locationSource", StringType(), True),
    StructField("magSource", StringType(), True),
])

In [4]:
from pyspark.sql.functions import col
df = spark.read.csv(
    "s3a://raw/earthquakes/year=2025/month=01/*.csv",
    header=True,
    schema=earthquake_schema,
)

df.groupBy("id").count().filter("count > 1").show()  # sanity check: any dupes before dedup?
df_clean = df.filter(col("id").isNotNull() & col("time").isNotNull())
df_deduped = df_clean.dropDuplicates(["id"])
print(f"before: {df.count()}, after dedup: {df_deduped.count()}")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/16 07:29:32 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://raw/earthquakes/year=2025/month=01/*.csv.
java.io.FileNotFoundException: No such file or directory: s3a://raw/earthquakes/year=2025/month=01/*.csv
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4118)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3976)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:3953)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:546)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:527)
	at org.ap

+---+-----+
| id|count|
+---+-----+
+---+-----+

before: 1276, after dedup: 1276


In [5]:
from pyspark.sql.functions import isnan

df.filter(isnan(col("mag")) | isnan(col("depth"))).show()

+----+--------+---------+-----+---+-------+---+---+----+---+---+---+-------+-----+----+---------------+----------+--------+------+------+--------------+---------+
|time|latitude|longitude|depth|mag|magType|nst|gap|dmin|rms|net| id|updated|place|type|horizontalError|depthError|magError|magNst|status|locationSource|magSource|
+----+--------+---------+-----+---+-------+---+---+----+---+---+---+-------+-----+----+---------------+----------+--------+------+------+--------------+---------+
+----+--------+---------+-----+---+-------+---+---+----+---+---+---+-------+-----+----+---------------+----------+--------+------+------+--------------+---------+



In [6]:
from pyspark.sql.functions import when, col, hour, dayofweek, lit

df_enriched = (
    df_deduped
    .withColumn(
        "mag_category",
        when(col("mag") < 4, "minor")
        .when(col("mag") < 5, "light")
        .when(col("mag") < 6, "moderate")
        .when(col("mag") < 7, "strong")
        .otherwise("major")
    )
    .withColumn(
        "depth_category",
        when(col("depth") < 70, "shallow")
        .when(col("depth") < 300, "intermediate")
        .otherwise("deep")
    )
    .withColumn(
        "is_reliable",
        (col("gap") < 180) & (col("rms") < 1.0) & (col("nst") >= 10)
    )
    .withColumn(
        "estimated_energy_joules",
        lit(10.0) ** (1.5 * col("mag") + 4.8)
    )
    .withColumn("hour_of_day", hour(col("time")))
    .withColumn("day_of_week", dayofweek(col("time")))  # 1 = Sunday ... 7 = Saturday
)

df_enriched.select("mag", "mag_category", "depth", "depth_category", "is_reliable", "estimated_energy_joules").show(5)

+----+------------+------+--------------+-----------+-----------------------+
| mag|mag_category| depth|depth_category|is_reliable|estimated_energy_joules|
+----+------------+------+--------------+-----------+-----------------------+
| 1.7|       minor|  10.5|       shallow|       NULL|    2.238721138568338E7|
| 1.8|       minor|8.0779|       shallow|       true|    3.162277660168379E7|
| 1.4|       minor|   8.4|       shallow|       NULL|      7943282.347242805|
|0.56|       minor|   9.9|       shallow|       true|      436515.8322401657|
| 5.1|    moderate|  10.0|       shallow|       true|   2.818382931264449E12|
+----+------------+------+--------------+-----------+-----------------------+
only showing top 5 rows


In [7]:
from pyspark.sql.functions import coalesce

df_enriched = df_enriched.withColumn(
    "is_reliable",
    col("is_reliable")
    & (col("depth") >= 0)
    & (col("mag") > -5) & (col("mag") < 10)
    & (col("latitude").between(-90, 90))
    & (col("longitude").between(-180, 180))
)

In [8]:
from pyspark.sql.functions import lpad, year, month, dayofmonth, col

df_final = (
    df_enriched
    .withColumn("year", year(col("time")))
    .withColumn("month", lpad(month(col("time")), 2, "0"))
    .withColumn("day", lpad(dayofmonth(col("time")), 2, "0"))
)

In [9]:
(
    df_final.write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet("s3a://processed/earthquakes")
)

26/09/16 07:29:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [10]:
jdbc_url = "jdbc:postgresql://localhost:5432/earthquake"
jdbc_properties = {
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

In [11]:
import psycopg2

conn = psycopg2.connect(
    host="localhost", port=5432, dbname="earthquake",
    user=os.environ["POSTGRES_USER"], password=os.environ["POSTGRES_PASSWORD"],
)
cur = conn.cursor()
cur.execute("""
    CREATE TABLE IF NOT EXISTS staging.earthquakes (
        id TEXT PRIMARY KEY,
        time TIMESTAMP,
        latitude DOUBLE PRECISION,
        longitude DOUBLE PRECISION,
        depth DOUBLE PRECISION,
        mag DOUBLE PRECISION,
        "magType" TEXT,
        place TEXT,
        type TEXT,
        status TEXT,
        net TEXT,
        updated TIMESTAMP,
        mag_category TEXT,
        depth_category TEXT,
        is_reliable BOOLEAN,
        estimated_energy_joules DOUBLE PRECISION,
        hour_of_day INTEGER,
        day_of_week INTEGER
    );
""")
conn.commit()
cur.close()
conn.close()

In [12]:
(
    df_final.select(
        "id", "time", "latitude", "longitude", "depth", "mag",
        "magType", "place", "type", "status", "net", "updated",
        "mag_category", "depth_category", "is_reliable",
        "estimated_energy_joules", "hour_of_day", "day_of_week"
    )
    .write
    .mode("overwrite")
    .jdbc(jdbc_url, "staging.earthquakes_scratch", properties=jdbc_properties)
)

In [13]:
conn = psycopg2.connect(
    host="localhost", port=5432, dbname="earthquake",
    user=os.environ["POSTGRES_USER"], password=os.environ["POSTGRES_PASSWORD"],
)
cur = conn.cursor()
cur.execute("""
    INSERT INTO staging.earthquakes
    SELECT * FROM staging.earthquakes_scratch
    ON CONFLICT (id) DO UPDATE SET
        time = EXCLUDED.time,
        latitude = EXCLUDED.latitude,
        longitude = EXCLUDED.longitude,
        depth = EXCLUDED.depth,
        mag = EXCLUDED.mag,
        "magType" = EXCLUDED."magType",
        place = EXCLUDED.place,
        type = EXCLUDED.type,
        status = EXCLUDED.status,
        net = EXCLUDED.net,
        updated = EXCLUDED.updated,
        mag_category = EXCLUDED.mag_category,
        depth_category = EXCLUDED.depth_category,
        is_reliable = EXCLUDED.is_reliable,
        estimated_energy_joules = EXCLUDED.estimated_energy_joules,
        hour_of_day = EXCLUDED.hour_of_day,
        day_of_week = EXCLUDED.day_of_week;
""")
conn.commit()
cur.close()
conn.close()

In [14]:
cur = psycopg2.connect(
    host="localhost", port=5432, dbname="earthquake",
    user=os.environ["POSTGRES_USER"], password=os.environ["POSTGRES_PASSWORD"],
).cursor()
cur.execute("SELECT COUNT(*) FROM staging.earthquakes;")
print(cur.fetchone())

(1276,)


In [15]:
df_check = spark.read.parquet("s3a://processed/earthquakes/year=2025/month=01/day=28")
df_check.show()
print(df_check.count())

+--------------------+--------+---------+-----+---+-------+----+----+----+----+---+------------+--------------------+--------------------+----------+---------------+----------+--------+------+--------+--------------+---------+------------+--------------+-----------+-----------------------+-----------+-----------+
|                time|latitude|longitude|depth|mag|magType| nst| gap|dmin| rms|net|          id|             updated|               place|      type|horizontalError|depthError|magError|magNst|  status|locationSource|magSource|mag_category|depth_category|is_reliable|estimated_energy_joules|hour_of_day|day_of_week|
+--------------------+--------+---------+-----+---+-------+----+----+----+----+---+------------+--------------------+--------------------+----------+---------------+----------+--------+------+--------+--------------+---------+------------+--------------+-----------+-----------------------+-----------+-----------+
|2025-01-28 00:19:...| 59.2149|-135.5408|  5.6|3.0|    

In [16]:
df_enriched.groupBy("is_reliable").count().show()

+-----------+-----+
|is_reliable|count|
+-----------+-----+
|       NULL|  293|
|       true|  624|
|      false|  359|
+-----------+-----+

